In [1]:
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [2]:
TRAIN_LOCS_KEY = 'train_locs'
TRAIN_IDS_KEY = 'train_ids'
TAXON_IDS_KEY = 'taxon_ids'
TAXON_NAME_KEY = 'taxon_names'

Reading the file:

In [3]:
filepath = os.path.join(os.getcwd(), 'species_train.npz')
data = np.load(filepath, allow_pickle=True)
train_locs = data[TRAIN_LOCS_KEY]
train_ids = data[TRAIN_IDS_KEY]
taxon_ids = data[TAXON_IDS_KEY]
taxon_names = data[TAXON_NAME_KEY]

Mapping the taxon ids to taxon latin names: 

In [4]:
species_ids_names = dict(zip(data['taxon_ids'], data['taxon_names']))  # latin names of species 

Create pandas Dataframe from the data: 

In [5]:
df = pd.DataFrame({
    'latitude': train_locs[:, 0],
    'longitude': train_locs[:, 1], 
    'taxon_id': data['train_ids']
})
df['taxon_name'] = [species_ids_names[id] for id in data['train_ids'].astype(int)]
df.head()

,latitude,longitude,taxon_id,taxon_name
0,-18.286728,143.481247,31529,Lophognathus gilberti
1,-13.099798,130.783646,31529,Lophognathus gilberti
2,-13.965274,131.695145,31529,Lophognathus gilberti
3,-12.853950,132.800507,31529,Lophognathus gilberti
4,-12.196790,134.279327,31529,Lophognathus gilberti


In [6]:
df.shape

(272037, 4)

Data Cleanining: 

<small>1. Check for missing or invalid coordinates:</small>

In [7]:
df = df.dropna(subset=['latitude', 'longitude'])
df = df[(df['latitude'].between(-90, 90)) & (df['longitude'].between(-180, 180))]
df.shape

(272037, 4)

<small>2. Remove any duplicates or nearly duplicates (observations that are extremely close):</small>

In [8]:
df['lat_rounded'] = df['latitude'].round(5)
df['lon_rounded'] = df['longitude'].round(5)

In [9]:
df = df.drop_duplicates(subset=['lat_rounded', 'lon_rounded'])
df.shape

(237977, 6)

<small>4. Validate species IDs: </small>

In [10]:
df['taxon_id'].isna().sum()

np.int64(0)

<small>5. Convert to categorical labels:</small>

In [11]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['taxon_id'])
df.head()

,latitude,longitude,taxon_id,taxon_name,lat_rounded,lon_rounded,label
0,-18.286728,143.481247,31529,Lophognathus gilberti,-18.28673,143.481247,278
1,-13.099798,130.783646,31529,Lophognathus gilberti,-13.09980,130.783646,278
2,-13.965274,131.695145,31529,Lophognathus gilberti,-13.96527,131.695145,278
3,-12.853950,132.800507,31529,Lophognathus gilberti,-12.85395,132.800507,278
4,-12.196790,134.279327,31529,Lophognathus gilberti,-12.19679,134.279327,278


<small>6. Only keep birds:</small>

<small>Note: Only run the next 2 blocks one time as they take a few seconds:</small>

In [12]:
taxa = pd.read_csv('taxa.csv')
birds = taxa[taxa['class'] == 'Aves']
bird_taxon_ids = set(birds['id'])
len(bird_taxon_ids)

35102

In [13]:
df = df[df['taxon_id'].isin(bird_taxon_ids)].copy()
df.shape

(151391, 7)

<small>Append the temprature data</small>

In [14]:
# Add the temprature data

<small>8. Split the data to x and y</small>

In [15]:
x_data = df.drop(columns=['taxon_id', 'taxon_name', 'lat_rounded', 'lon_rounded', 'label'])
y_data = df['label']

<small>9. Split to train and validation sets</small>

Exploratory Data Analysis:

In [16]:
# Hajer

Models: